# Your scraper has an accent

Two clients. Same machine, same address, same URL. One gets a block page and
the other gets the site.

The only difference between them is what their TLS handshake and HTTP/2
connection look like on the wire. No proxy, no JavaScript, no captcha solver.

In [ ]:
import html
import json

import httpx
from IPython.display import HTML, display

from pybrws.client import get

TARGET = "https://www.zillow.com/"


def show(label, status, page, height=420):
    """Render a fetched page in a sandboxed frame, scripts disabled."""
    size = f"{len(page):,} bytes"
    tone = "#1a7f37" if status == 200 else "#cf222e"
    frame = (
        f'<iframe sandbox style="width:100%;height:{height}px;border:0" '
        f'srcdoc="{html.escape(page, quote=True)}"></iframe>'
    )
    display(HTML(
        f'<div style="font:600 14px system-ui;margin:4px 0 8px">{label} '
        f'<span style="color:{tone}">HTTP {status}</span> '
        f'<span style="color:#656d76;font-weight:400">{size}</span></div>'
        f'<div style="border:1px solid #d0d7de;border-radius:6px;overflow:hidden">'
        f'{frame}</div>'
    ))

## 1. A mature HTTP/2 client

`httpx` over Python's standard TLS stack. Correct, maintained, widely used.

In [ ]:
with httpx.Client(http2=True, follow_redirects=True, timeout=30) as client:
    baseline = client.get(TARGET)

show("httpx", baseline.status_code, baseline.text)

## 2. The same request, wearing Firefox's fingerprint

Identical URL, identical machine. `pybrws` differs only in the bytes of its
ClientHello, its HTTP/2 connection preamble and its header order.

In [ ]:
ours = get(TARGET, follow_redirects=5, timeout=30)

show("pybrws", ours.status, ours.text())

## 3. What the server actually saw

`tls.peet.ws` reports back the fingerprint of whoever asked. The reference row
is a real Firefox 157, captured by opening that URL in the browser.

In [ ]:
PROBE = "https://tls.peet.ws/api/all"
REFERENCE = json.loads(
    open("fixtures/reference-firefox-157-peet.json", encoding="utf-8").read()
)

with httpx.Client(http2=True, timeout=30) as client:
    seen_baseline = client.get(PROBE).json()
seen_ours = json.loads(get(PROBE, timeout=30).body)

rows = [
    ("Firefox 157", REFERENCE["tls"]["ja4"], REFERENCE["http2"]["akamai_fingerprint"]),
    ("pybrws", seen_ours["tls"]["ja4"], seen_ours["http2"]["akamai_fingerprint"]),
    ("httpx", seen_baseline["tls"]["ja4"], seen_baseline["http2"]["akamai_fingerprint"]),
]

cells = "".join(
    '<tr><td style="padding:6px 14px 6px 0"><b>{}</b></td>'
    '<td style="padding:6px 14px 6px 0;color:{}">{}</td>'
    '<td style="padding:6px 0;color:{}">{}</td></tr>'.format(
        name,
        "#1a7f37" if ja4 == rows[0][1] else "#cf222e", ja4,
        "#1a7f37" if akamai == rows[0][2] else "#cf222e", akamai,
    )
    for name, ja4, akamai in rows
)
display(HTML(
    '<table style="font:13px ui-monospace,monospace;border-collapse:collapse">'
    '<tr style="text-align:left;border-bottom:1px solid #d0d7de">'
    '<th style="padding:6px 14px 6px 0">client</th><th>JA4</th>'
    '<th style="padding-left:14px">HTTP/2 fingerprint</th></tr>'
    + cells + '</table>'
))

## 4. The part that is easy to oversell

The fingerprint is necessary and it is not sufficient. Crossing it against the
address it comes from, three rounds each:

| | pybrws | httpx |
|---|---|---|
| **home connection** | 3/3 served | 0/3 |
| **commercial ISP proxy** | 0/3 | 0/3 |

One cell of four passes. A proxy address sold as clean is itself a signal, and
here it scored worse than an ordinary domestic line.

And nothing here solves a JavaScript challenge. A site that issues one is out
of reach for any HTTP client, by construction.